In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate


In [3]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = loader.load()
len(docs)

10

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 1000 , chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

11

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5165.97it/s]


In [6]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [ ]:
query = "Machine Learning and Data sciece content"
data = vector_store.similarity_search(query=query)


'🎤  Mock Interview: Stats Scenarios \ue09d Probabilities \ue09d Tests\n✅  Module 6: Machine Learning – I (Supervised \nLearning) (4 weeks)\nDuration: Month 6\nTopics:\nML pipeline\nRegression: Linear, Logistic\nDecision Tree, Random Forest, KNN\nTrain-test split, model evaluation\nTools:\nScikit-learn, Google Colab, ChatGPT, PyCaret (optional)\nMini Project:\nLoan Approval or House Price Prediction\nPredict Diabetes from health dataset\n🎤  Mock Interview: Supervised Learning Models \ue09d Metrics\n✅  Module 7: Machine Learning – II (Unsupervised & \nFeature Engineering) (3 weeks)\nDuration: Month 7\nTopics:\nKMeans Clustering\nDimensionality Reduction: PCA\nFeature selection, encoding, scaling\nModel tuning \ue081GridSearchCV\ue082\nTools:\nScikit-learn, Seaborn, Colab\nMini Project:\n📘  1\ue088 Y e ar R o admap: Dat a Anal y tics, Dat a Science & GenAI\n5'

In [12]:
context = ""
for doc in data:
    context += doc.page_content + "\n"



In [14]:
llm = ChatGroq(model="openai/gpt-oss-20b")


In [ ]:
## Chain - Context_generate | prompt | llm | strparser




AIMessage(content='**Machine Learning & Data‑Science Highlights from the 1‑Year Roadmap**\n\n| Module | Duration | Core Topics | Key Tools | Mini‑Project | Mock‑Interview Focus |\n|--------|----------|-------------|-----------|--------------|----------------------|\n| **Module\u202f6 – Machine Learning\u202fI (Supervised Learning)** | 4\u202fweeks (Month\u202f6) | • ML pipeline fundamentals<br>• Regression (Linear & Logistic)<br>• Decision Tree, Random Forest, KNN<br>• Train‑test split & model evaluation | Scikit‑learn, Google\u202fColab, PyCaret (optional), ChatGPT | • Loan approval or house‑price prediction<br>• Predict diabetes from a health dataset | • Metrics (MAE, RMSE, ROC‑AUC, confusion matrix)<br>• Model evaluation & tuning |\n| **Module\u202f7 – Machine Learning\u202fII (Unsupervised & Feature Engineering)** | 3\u202fweeks (Month\u202f7) | • KMeans clustering<br>• Dimensionality reduction (PCA)<br>• Feature selection, encoding, scaling<br>• Hyper‑parameter tuning (GridSearchC

In [17]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context += doc.page_content + "\n"
    
    return {
        "context":context,
        "question":query
    }

In [18]:
prompt = PromptTemplate.from_template("""You are a helpful assistant  and provide answers based on the context for user question and if you don't know the answer then you can say that 'I don't know'
    Context : {context}
    
    Question:{question}                              
                                      """)

In [19]:
rag_chain = get_context | prompt | llm

In [21]:
res = rag_chain.invoke("What is the duration of my module 3?  ")
res.content

'Module\u202f3 (“SQL & Excel for Analysts”) runs for **3 weeks** (i.e., it is scheduled in **Month\u202f3** of the roadmap).'